# Unidad 6 - Temas principales

Notebook con los 4 temas del programa:
1. Limpieza e Integración
2. Medidas de Tendencia y Dispersión
3. Distribuciones y Correlación
4. Transformación y Reducción

Dataset utilizado: **bank-full.csv** (clientes de un banco con info de saldo, trabajo, préstamo, etc.)

In [ ]:
# Imports y configuracion de Colab
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

from google.colab import drive
import os
drive.mount('/content/gdrive')
os.chdir('/content/gdrive/My Drive')


In [ ]:
# Carga del dataset principal
# bank-full.csv: encuesta de clientes de un banco portugues
# Variables clave: balance (saldo), job, marital, education, loan
bank = pd.read_csv('bank-full.csv')
print('Shape:', bank.shape)
bank.head()


# 1. Limpieza e Integración

**Limpieza de datos**: proceso de detectar y corregir errores, valores faltantes e inconsistencias en el dataset.

**Integración**: combinación de datos provenientes de distintas fuentes en un único DataFrame.

Sin una buena limpieza los análisis posteriores (estadísticas, modelos) pueden arrojar resultados incorrectos.

In [ ]:
# 1.1 Exploración inicial: detectar valores nulos y tipos de datos
print('Shape:', bank.shape)
print()
print('Valores nulos por columna:')
print(bank.isnull().sum())
print()
print('Tipos de datos:')
print(bank.dtypes)


In [ ]:
# 1.2 Limpieza: eliminar duplicados y filas con nulos (si existieran)
print('Duplicados antes:', bank.duplicated().sum())
bank_clean = bank.drop_duplicates()
print('Duplicados despues:', bank_clean.duplicated().sum())

# Rellenar nulos numericos con la mediana (robusto frente a outliers)
# Rellenar nulos categoricos con la moda
for col in bank_clean.columns:
    if bank_clean[col].dtype == 'object':
        bank_clean[col].fillna(bank_clean[col].mode()[0], inplace=True)
    else:
        bank_clean[col].fillna(bank_clean[col].median(), inplace=True)

print('Nulos restantes:', bank_clean.isnull().sum().sum())


In [ ]:
# 1.3 Integracion: agregar una columna calculada como nueva 'fuente' de datos
# balance_category: clasifica a cada cliente segun su saldo (variable derivada)
bank_clean['balance_category'] = pd.cut(
    bank_clean['balance'],
    bins=[-float('inf'), 0, 1000, 5000, float('inf')],
    labels=['Negativo', 'Bajo', 'Medio', 'Alto']
)
print(bank_clean[['balance', 'balance_category']].head(10))
print()
print('Distribucion de categorias de saldo:')
print(bank_clean['balance_category'].value_counts())


# 2. Medidas de Tendencia y Dispersión

**Medidas de tendencia central**: describen el valor 'tipico' de la distribución.
- **Media**: promedio aritmético. Sensible a outliers.
- **Mediana**: valor central de los datos ordenados. Robusta frente a outliers.
- **Moda**: valor más frecuente. Útil en variables categóricas.

**Medidas de dispersión**: describen qué tan esparcidos están los datos.
- **Varianza**: promedio de los cuadrados de las desviaciones respecto a la media.
- **Desviación estándar (std)**: raíz de la varianza, en las mismas unidades que los datos.
- **IQR (Rango intercuartílico)**: Q3 - Q1. Robusto frente a outliers.
- **Percentiles**: dividen los datos en 100 partes iguales.

In [ ]:
# 2.1 Medidas de tendencia central sobre el saldo bancario
media    = bank_clean['balance'].mean()
mediana  = bank_clean['balance'].median()
moda     = bank_clean['balance'].mode()[0]

print(f'Media:   {media:.2f}')
print(f'Mediana: {mediana:.2f}')
print(f'Moda:    {moda:.2f}')
print()
# Diferencia media vs mediana: si media >> mediana -> distribucion sesgada a la derecha (outliers altos)
print(f'Diferencia media-mediana: {media - mediana:.2f}')


In [ ]:
# 2.2 Medidas de dispersion
varianza = bank_clean['balance'].var()
std      = bank_clean['balance'].std()
Q1       = bank_clean['balance'].quantile(0.25)
Q3       = bank_clean['balance'].quantile(0.75)
IQR      = Q3 - Q1

print(f'Varianza:          {varianza:.2f}')
print(f'Desv. estandar:    {std:.2f}')
print(f'Q1 (percentil 25): {Q1:.2f}')
print(f'Q3 (percentil 75): {Q3:.2f}')
print(f'IQR:               {IQR:.2f}')
print(f'Rango completo:    {bank_clean["balance"].min():.2f} a {bank_clean["balance"].max():.2f}')


In [ ]:
# 2.3 Boxplot: visualiza tendencia central y dispersion en un solo grafico
# La caja va de Q1 a Q3 (IQR), la linea del medio es la mediana
# Los bigotes llegan hasta 1.5*IQR; los puntos fuera son posibles outliers
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
sns.boxplot(y=bank_clean['balance'], color='lightblue')
plt.title('Boxplot - Balance (todos los clientes)')
plt.ylabel('Saldo ($)')

plt.subplot(1, 2, 2)
# Comparacion de dispersion del balance por nivel educativo
sns.boxplot(x='education', y='balance', data=bank_clean, palette='pastel')
plt.title('Balance por nivel educativo')
plt.xlabel('Educacion')
plt.ylabel('Saldo ($)')

plt.tight_layout()
plt.show()


# 3. Distribuciones y Correlación

**Distribución**: muestra cómo se distribuyen los valores de una variable.
- **Normal (Gaussiana)**: forma de campana, simétrica. Media = Mediana = Moda.
- **Sesgada a la derecha**: cola larga hacia valores altos (ej: salarios, saldos bancarios).
- **Uniforme**: todos los valores tienen igual probabilidad.

**Correlación**: mide la fuerza y dirección de la relación lineal entre dos variables numéricas.
- **r = 1**: correlación positiva perfecta.
- **r = -1**: correlación negativa perfecta.
- **r ≈ 0**: sin relación lineal.

> Importante: **correlación ≠ causalidad**.

In [ ]:
# 3.1 Distribucion del saldo bancario
# kde=True: superpone la curva de densidad estimada (Kernel Density Estimation)
# Permite ver la forma de la distribucion mas alla de las barras del histograma
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
sns.histplot(bank_clean['balance'], kde=True, bins=50, color='steelblue')
plt.axvline(media, color='red', linestyle='--', label=f'Media: {media:.0f}')
plt.axvline(mediana, color='green', linestyle='--', label=f'Mediana: {mediana:.0f}')
plt.title('Distribucion del Saldo Bancario')
plt.xlabel('Saldo ($)')
plt.legend()

# Comparacion por genero (loan yes/no)
plt.subplot(1, 2, 2)
for loan_val, color in [('yes', 'salmon'), ('no', 'lightblue')]:
    sns.histplot(bank_clean[bank_clean['loan'] == loan_val]['balance'],
                 kde=True, bins=40, color=color, alpha=0.5, label=f'Loan={loan_val}')
plt.title('Distribucion del Saldo por Prestamo')
plt.xlabel('Saldo ($)')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# 3.2 Matriz de correlacion entre variables numericas
# Solo aplica a variables numericas: .select_dtypes(include='number')
# Valores cercanos a 1 o -1 indican fuerte relacion lineal
# Valores cercanos a 0 indican independencia lineal
numeric_cols = bank_clean.select_dtypes(include='number')
corr_matrix = numeric_cols.corr()

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Matriz de Correlacion - Variables Numericas')
plt.tight_layout()
plt.show()


In [ ]:
# 3.3 Scatter plot: visualiza la relacion entre dos variables numericas
# hue='loan': colorea los puntos segun si el cliente tiene prestamo o no
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=bank_clean.sample(1000, random_state=42),  # muestra para no saturar el grafico
    x='age', y='balance',
    hue='loan', alpha=0.6, palette='Set1'
)
plt.title('Relacion entre Edad y Saldo Bancario')
plt.xlabel('Edad')
plt.ylabel('Saldo ($)')
plt.show()

# Coeficiente de correlacion de Pearson entre edad y saldo
corr_age_balance = bank_clean['age'].corr(bank_clean['balance'])
print(f'Correlacion Pearson (edad vs saldo): {corr_age_balance:.4f}')


# 4. Transformación y Reducción

**Transformación**: convierte variables a un formato más adecuado para el análisis o modelado.
- **Normalización (MinMaxScaler)**: escala valores al rango [0, 1]. Sensible a outliers.
- **Estandarización (StandardScaler)**: transforma a media=0 y std=1. Más robusta.
- **Codificación (LabelEncoder / OneHotEncoder)**: convierte categorías en números.

**Reducción de dimensionalidad (PCA)**:
- Técnica que reduce la cantidad de variables conservando la mayor parte de la información.
- Crea nuevas variables llamadas **Componentes Principales (PC)** que son combinaciones lineales de las originales.
- Útil para visualizar datos de alta dimensión y reducir ruido en el modelo.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA

# 4.1 Estandarizacion y Normalizacion del saldo
# StandardScaler: (x - media) / std  -> media=0, std=1
# MinMaxScaler:   (x - min) / (max - min)  -> rango [0, 1]
scaler_std   = StandardScaler()
scaler_minmax = MinMaxScaler()

balance_vals = bank_clean[['balance']]
balance_std    = scaler_std.fit_transform(balance_vals)
balance_minmax = scaler_minmax.fit_transform(balance_vals)

print('Balance original  - media:', round(bank_clean['balance'].mean(), 2),
      ' std:', round(bank_clean['balance'].std(), 2))
print('Estandarizado     - media:', round(balance_std.mean(), 4),
      ' std:', round(balance_std.std(), 4))
print('Normalizado [0,1] - min:', round(balance_minmax.min(), 4),
      ' max:', round(balance_minmax.max(), 4))


In [ ]:
# 4.2 Codificacion de variables categoricas con LabelEncoder
# LabelEncoder asigna un entero a cada categoria: 'primary'->0, 'secondary'->1, etc.
# Util para modelos que requieren inputs numericos
le = LabelEncoder()
bank_encoded = bank_clean.copy()
categorical_cols = bank_encoded.select_dtypes(include='object').columns

for col in categorical_cols:
    bank_encoded[col] = le.fit_transform(bank_encoded[col].astype(str))

print('Columnas codificadas:')
print(bank_encoded[categorical_cols].head())


In [ ]:
# 4.3 Reduccion de dimensionalidad con PCA
# PCA: busca las direcciones de maxima varianza en los datos
# n_components=2: reduce todas las variables numericas a solo 2 dimensiones
# Permite visualizar en 2D un dataset con muchas variables

# Seleccionamos columnas numericas (ya codificadas)
X = bank_encoded.select_dtypes(include='number').drop(columns=['balance'], errors='ignore')

# Estandarizamos antes de PCA (obligatorio: PCA es sensible a la escala)
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print('Varianza explicada por cada componente:')
for i, v in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {v*100:.1f}%')
print(f'  Total: {pca.explained_variance_ratio_.sum()*100:.1f}%')

# Visualizacion en el espacio PCA
df_pca = pd.DataFrame(X_pca, columns=['PC1', 'PC2'])
df_pca['loan'] = bank_clean['loan'].values

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df_pca.sample(1000, random_state=42),
                x='PC1', y='PC2', hue='loan', alpha=0.5, palette='Set1')
plt.title('Clientes en el espacio PCA (2 componentes)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% varianza)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% varianza)')
plt.show()
